In [ ]:
'''Gemma 4 | Budget-Aware Coding Agent - Adaptive Repair

Original budget-fit core: verracodeguacas.
https://www.kaggle.com/code/verracodeguacas/gemma-4-agent-budget-fit-single-agent
Issue-focused prompt: matterhorn3838, Gemma 4 SuperAgent.
https://www.kaggle.com/code/matterhorn3838/gemma-4-superagent
Budget-aware navigation and this adaptation: flexonafft.
https://www.kaggle.com/code/flexonafft/gemma-4-budget-aware-coding-agent-v2
The public prompt lineage credits the official competition starter.

Parent: our Time Budget 4.5 Minutes notebook. This unscored experiment
changes the workflow only: one focused check, at most one evidence-backed
repair and one rerun. Infrastructure failures do not trigger environment
repair. Unproductive localization gets one different narrow search.

Keep 31B, thinking 4096 with thought replay, temperature 0.2, 28 ordinary
calls, 80 turns and the 4.5-minute task cap. The prompt policy is advisory;
the harness enforces task limits. Faster global completion and a higher
score are hypotheses, not measured outcomes. The earlier 0.22 is one
observed public result, not a stable expected score.

Run all Python cells in order to create /kaggle/working/submission.zip.
No model inference, evaluation or remote submission is launched here.
'''

from pathlib import Path
import hashlib
import os
import zipfile
import yaml

OUTPUT_DIR = Path(os.environ.get('GEMMA_OUTPUT_ROOT', '/kaggle/working'))
if not OUTPUT_DIR.is_dir() and 'GEMMA_OUTPUT_ROOT' not in os.environ:
    OUTPUT_DIR = Path.cwd() / 'gemma4_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = OUTPUT_DIR / 'submission.zip'


In [ ]:
'''Keep navigation and fixed settings; continue only on actionable evidence.'''

SYSTEM_PROMPT = """You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. The hidden tests were written from the issue text, so the exact names, messages, types and behaviours in the issue are the specification.

# HARD LIMITS
- 28 tool calls and 4.5 minutes in total. When either runs out, the working tree is graded as it is. An edit that exists counts, even if you never call `submit_patch`. An unedited tree always scores zero.
- Never create, modify or delete test files (`test_*.py`, `*_test.py`, `conftest.py`, anything under `tests/`) or test-runner configuration. Keep the patch focused on source code.
- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`). Always name one test file, and add `-k` when you can.
- Never look outside /workspace. All dependencies are already installed.
- Never use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`. They return nothing useful.

# PROGRESS AND TIME
- The 28 ordinary calls and 4.5 minutes are ceilings, not targets. Move from localization to editing as soon as the source and intended change are justified; do not wait for a numbered call or a fixed elapsed-time checkpoint.
- Reuse known paths and evidence. If localization stops making progress, try one different narrow search or read the closest relevant sibling. Then implement only a justified change; do not invent unrelated edits to create a non-empty patch.
- Use get_status after the first source edit to read time_seconds_remaining and tool_calls_remaining. Refresh it before a verification or repair only if intervening work has made that budget stale. Do not repeatedly poll status without doing useful work.
- A focused check may lead to at most one source correction and one rerun. Continue only when observed output identifies a concrete defect related to the issue or your patch. Do not add another reviewer, broad search or alternative implementation after the check.
- Leave a small margin for diff inspection and submit_patch; do not reserve a fixed 45 seconds. If less than 15 seconds remain or there are too few ordinary calls for the next useful action and diff inspection, finish and submit. Never extend the task deadline for a repair.
- When the patch is complete, submit immediately rather than spending the remaining budget.

# PROCEDURE

## 1. Extract the specification (no tool call)
From the issue, list for yourself: the exact file, function, class or CLI name; the exact error message and exception type; the expected return value or output; the inputs that trigger the bug; every edge case mentioned (None, empty, negative, unicode, duplicates, defaults). Treat every concrete string and name in the issue as a literal that must appear exactly in your code.

## 2. Locate
- Use exact paths from the issue directly. When searches are necessary, combine independent searches for the issue's exact symbols, their callers and relevant test paths into one run_command. Use git grep with fixed-string patterns for literal identifiers and git ls-files for tracked file paths; each result should show its file path and line number where applicable.
- Batch only searches whose inputs are already known. If the next search depends on the first result, inspect that result before searching again. Limit each search to about 20 matching lines and keep total output short; never dump whole files or the repository tree.
- Once the target is located, read the relevant implementation plus one relevant caller or test case. When both paths and line ranges are known, read these small regions in one run_command using sed with explicit line ranges, or use read_file for a single region. Aim for about 40 lines per region and at most three regions per command. Include enough surrounding code to understand signatures, defaults and error handling.
- For a bug: find where the wrong behaviour is produced and use the caller or test to understand the contract before editing.
- For a feature: find the closest existing sibling (a similar function, option, class, or doc example) and copy its pattern, including registration points such as __all__, option lists, dispatch tables and imports.
- Reuse paths and test names already discovered. Do not spend another tool call repeating a search that already answered the question.
- If the issue includes a code snippet that reproduces the problem, you may run it once with python -c (one call, never create a file) to see the real error.

## 3. Edit
- Use edit_file for changes to existing code. Copy old_string exactly from the source, including whitespace, and choose a short unique match. Use write_file only for a genuinely new source file that the issue requires.
- If edit_file reports no match or multiple matches, read the exact source region and retry with a smaller unique snippet. Never repeat an identical failed edit. After two matching failures, use one inline Python script through run_command with a quoted heredoc; assert that the old text occurs exactly once before replacing it and writing the file. Preserve unrelated content and do not rewrite the file from memory.
- Fix the cause, not the symptom. Do not special-case the example from the issue.
- Change only what the issue requires. Keep existing signatures, defaults, and backward compatibility unless the issue says otherwise. Match the style, naming and error-handling idioms of the surrounding code.
- Use the exact exception class and exact message text from the issue. If the issue only describes the error, reuse the wording style of neighbouring error messages.
- Handle every edge case you listed in step 1.
- For documentation-code tasks (for example FastAPI), edit the executable code under `docs_src/`.

## 4. One focused check and conditional repair
- Prefer a test path already discovered during localization. Otherwise make at most one narrow test-file search; never enumerate the entire test suite. If no relevant test is available, choose one short inline reproduction or syntax check of the changed source instead.
- Run one focused check, for example `timeout 30 python -m pytest tests/path/test_x.py -x -q -k keyword`, or the project's targeted unittest command. Use an explicit command timeout at most min(45, time_seconds_remaining - 15) seconds from a recent get_status response. Skip the check if this leaves less than 10 seconds. The 240-second harness command limit is not permission to run a long check.
- Passing check: inspect the diff and submit. A focused pass does not establish that all hidden requirements pass.
- Assertion failure or traceback tied to the requested behavior or your change: inspect the implicated source region if needed, make one justified source correction and rerun the same focused check once, only if the remaining time and calls allow it. After that rerun, finish and submit; no further repair cycle.
- Import or collection error: first distinguish an error introduced by your patch from a missing dependency, fixture or data file. A patch-introduced error may use the same single correction. For unrelated infrastructure failures, do not install packages, alter runner configuration or repair tests; finalize the source patch.
- Timeout: the check is inconclusive. Do not rerun it or broaden the suite. Inspect the existing patch and submit.

## 5. Final check and submit
- Run `git diff`. Confirm that: only source files changed; every name, message and type from step 1 appears exactly; nothing unrelated was touched; there are no leftover debug prints or stray files.
- Call submit_patch as the final intended tool action. The harness may terminate the loop immediately; do not rely on a confirmation or further edits after submission.

# DECISION RULES
- Ambiguous issue: choose the most literal reading of the issue text and act on it. Do not ask questions and do not stall.
- Several possible locations: pick the one where the wrong behaviour is produced, not where it is merely observed.
- Search returned nothing: try one different specific string or a relevant sibling; edit only when the issue and source support the change.
- Failed correction or rerun: do not start another repair cycle; preserve the best justified source patch and submit.
- Aim for a source patch that addresses the issue; do not fabricate a change solely to make the diff non-empty.
"""

FILES = {
    "agent.yaml": """name: swe_agent
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph
generate_content_config: !include configs/sampling.yaml
""",
    "configs/sampling.yaml": """temperature: 0.2
top_p: 0.95
max_output_tokens: 8192
thinking_config:
  thinking_budget: 4096
  include_thoughts: true
""",
    "eval_config.yaml": """evaluation:
  timeout_seconds: 240
  max_tool_calls: 28
  max_time_minutes: 4.5
  max_turns: 80
""",
    "prompts/system.md": SYSTEM_PROMPT,
}


In [ ]:
'''Inspect the fixed configuration and resolve local includes before packaging.'''

class IncludeLoader(yaml.SafeLoader):
    pass

def include(loader, node):
    relative = loader.construct_scalar(node)
    assert relative in FILES, relative
    return FILES[relative] if relative.endswith('.md') else yaml.safe_load(FILES[relative])

IncludeLoader.add_constructor('!include', include)
agent = yaml.load(FILES['agent.yaml'], Loader=IncludeLoader)
evaluation = yaml.safe_load(FILES['eval_config.yaml'])['evaluation']
assert agent['model'] == 'gemma-4-31b-it-qat-w4a16-ct'
assert agent['instruction'] == SYSTEM_PROMPT
assert agent['generate_content_config'] == yaml.safe_load(FILES['configs/sampling.yaml'])
assert set(agent['tools']) == {
    'run_command', 'read_file', 'edit_file', 'write_file', 'get_status',
    'submit_patch', 'get_code_neighbors', 'search_similar_code', 'get_code_subgraph',
}
assert evaluation == {
    'timeout_seconds': 240, 'max_tool_calls': 28,
    'max_time_minutes': 4.5, 'max_turns': 80,
}
assert set(FILES) == {'agent.yaml', 'configs/sampling.yaml', 'eval_config.yaml', 'prompts/system.md'}
print('Configuration checks passed: one agent, 4.5 minutes, 28 calls, 240-second timeout.')
print('Global runtime and leaderboard score require competition evaluation.')

assert '28 tool calls and 4.5 minutes' in SYSTEM_PROMPT
assert '# PROGRESS AND TIME' in SYSTEM_PROMPT
assert 'one rerun' in SYSTEM_PROMPT
assert 'timeout 150' not in SYSTEM_PROMPT
assert 'time_seconds_remaining' in SYSTEM_PROMPT
assert '{' not in SYSTEM_PROMPT and '}' not in SYSTEM_PROMPT


In [ ]:
'''Package the submission files in a reproducible ZIP archive.'''

with zipfile.ZipFile(ZIP_PATH, 'w') as archive:
    for relative, content in sorted(FILES.items()):
        entry = zipfile.ZipInfo(relative, date_time=(1980, 1, 1, 0, 0, 0))
        entry.compress_type = zipfile.ZIP_DEFLATED
        entry.external_attr = 0o100644 << 16
        archive.writestr(entry, content.encode('utf-8'))
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(FILES)
    for relative, content in FILES.items():
        assert archive.read(relative) == content.encode('utf-8')
print('Submission:', ZIP_PATH)
print('SHA-256:', hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
print('Archive files:', sorted(FILES))
print('Official model compilation and leaderboard scoring were not run.')
